<a href="https://colab.research.google.com/github/vaishnavikabbe/AIML/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vaishnavikabbe/AIML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My baseline rule

I will prioritize pages that show stronger evidence that they may need content review. The baseline score will combine observed search-performance and content signals rather than relying on a single feature.

Pages will receive points when they have lower recent impressions, lower search performance relative to demand, or other measurable signals that suggest they may deserve review.

The score is a baseline decision-support rule, not a prediction of future Google rankings.

### Reason codes

- LOW_IMPRESSIONS: the page has relatively low 90-day impressions.
- LOW_SEARCH_DEMAND: the page has relatively low observed search volume.
- CONTENT_SIGNAL: the page has a content-related signal that makes it worth reviewing.
- REVIEW_PRIORITY: the page has multiple signals and therefore receives a higher overall score.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-07 Section 2
# Build baseline action score and ranked review queue

import os
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Make sure the repository exists
# --------------------------------------------------

REPO_DIR = "/content/AIML"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/vaishnavikabbe/AIML.git /content/AIML

os.chdir(REPO_DIR)

# --------------------------------------------------
# 2. Load dataset
# --------------------------------------------------

DATA_PATH = "data/raw/content_refresh_anonymized.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}"
    )

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

# --------------------------------------------------
# 3. Check required columns
# --------------------------------------------------

required_columns = [
    "search_volume",
    "impressions_90d"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

# --------------------------------------------------
# 4. Convert numeric columns
# --------------------------------------------------

df["search_volume"] = pd.to_numeric(
    df["search_volume"],
    errors="coerce"
)

df["impressions_90d"] = pd.to_numeric(
    df["impressions_90d"],
    errors="coerce"
)

# --------------------------------------------------
# 5. Fill missing values using medians
# --------------------------------------------------

df["search_volume"] = df["search_volume"].fillna(
    df["search_volume"].median()
)

df["impressions_90d"] = df["impressions_90d"].fillna(
    df["impressions_90d"].median()
)

# --------------------------------------------------
# 6. Create percentile ranks
# --------------------------------------------------

df["search_volume_pct"] = (
    df["search_volume"].rank(pct=True)
)

df["impressions_pct"] = (
    df["impressions_90d"].rank(pct=True)
)

# --------------------------------------------------
# 7. Create a simple action score
# --------------------------------------------------
#
# Higher score = stronger reason for review.
#
# Low impressions receive higher priority.
# Low search volume receives a smaller priority component.
#
# This is deliberately simple and explainable.

df["low_impressions_score"] = (
    1 - df["impressions_pct"]
)

df["low_search_score"] = (
    1 - df["search_volume_pct"]
)

df["baseline_score"] = (
    0.7 * df["low_impressions_score"]
    + 0.3 * df["low_search_score"]
)

# --------------------------------------------------
# 8. Add reason codes
# --------------------------------------------------

df["reason_code"] = np.where(
    df["baseline_score"] >= df["baseline_score"].quantile(0.90),
    "REVIEW_PRIORITY",
    np.where(
        df["impressions_pct"] <= 0.25,
        "LOW_IMPRESSIONS",
        np.where(
            df["search_volume_pct"] <= 0.25,
            "LOW_SEARCH_DEMAND",
            "GENERAL_REVIEW"
        )
    )
)

# --------------------------------------------------
# 9. Rank pages
# --------------------------------------------------

df = df.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

df["rank"] = df.index + 1

# --------------------------------------------------
# 10. Create output
# --------------------------------------------------

output_columns = [
    "rank",
    "baseline_score",
    "reason_code",
    "search_volume",
    "impressions_90d"
]

# Keep only columns that actually exist
output_columns = [
    col for col in output_columns
    if col in df.columns
]

output = df[output_columns].copy()

OUTPUT_PATH = "work/outputs/baseline_action_score.csv"

os.makedirs(
    "work/outputs",
    exist_ok=True
)

output.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nBaseline queue created successfully.")
print("Output:", OUTPUT_PATH)
print("Rows in queue:", len(output))

print("\nTop 10:")
print(output.head(10).to_string(index=False))

Dataset loaded successfully.
Rows: 30000
Columns: 44

Baseline queue created successfully.
Output: work/outputs/baseline_action_score.csv
Rows in queue: 30000

Top 10:
 rank  baseline_score     reason_code  search_volume  impressions_90d
    1        0.932037 REVIEW_PRIORITY            0.0                1
    2        0.932037 REVIEW_PRIORITY            0.0                1
    3        0.932037 REVIEW_PRIORITY            0.0                1
    4        0.932037 REVIEW_PRIORITY            0.0                1
    5        0.932037 REVIEW_PRIORITY            0.0                1
    6        0.932037 REVIEW_PRIORITY            0.0                1
    7        0.932037 REVIEW_PRIORITY            0.0                1
    8        0.932037 REVIEW_PRIORITY            0.0                1
    9        0.932037 REVIEW_PRIORITY            0.0                1
   10        0.932037 REVIEW_PRIORITY            0.0                1


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

The Top-20 pages are treated as a review queue rather than confirmed problems. Each page receives an action based on its baseline reason code.

The confidence note explains that the score is directional and based only on the available observed signals. A reviewer should check the page before taking action.

A recommendation could be wrong if the page has an important business purpose, temporary traffic variation, data-quality issues, or another context that is not represented in the dataset.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-07 Section 3
# Generate the Top-20 review table

top20 = output.head(20).copy()

def get_action(reason):
    if reason == "LOW_IMPRESSIONS":
        return "Review recent content and search performance"
    elif reason == "LOW_SEARCH_DEMAND":
        return "Review whether the page still matches search demand"
    elif reason == "REVIEW_PRIORITY":
        return "Prioritize for manual content review"
    else:
        return "Review page before making changes"

top20["action"] = top20["reason_code"].apply(get_action)

top20["confidence_note"] = (
    "Directional baseline; manual review required"
)

top20["what_could_make_it_wrong"] = (
    "Temporary trend, data quality issue, or missing business context"
)

print("TOP-20 REVIEW QUEUE")
print("=" * 100)

print(
    top20[
        [
            "rank",
            "baseline_score",
            "reason_code",
            "action",
            "confidence_note",
            "what_could_make_it_wrong"
        ]
    ].to_string(index=False)
)

TOP-20 REVIEW QUEUE
 rank  baseline_score     reason_code                               action                              confidence_note                                         what_could_make_it_wrong
    1        0.932037 REVIEW_PRIORITY Prioritize for manual content review Directional baseline; manual review required Temporary trend, data quality issue, or missing business context
    2        0.932037 REVIEW_PRIORITY Prioritize for manual content review Directional baseline; manual review required Temporary trend, data quality issue, or missing business context
    3        0.932037 REVIEW_PRIORITY Prioritize for manual content review Directional baseline; manual review required Temporary trend, data quality issue, or missing business context
    4        0.932037 REVIEW_PRIORITY Prioritize for manual content review Directional baseline; manual review required Temporary trend, data quality issue, or missing business context
    5        0.932037 REVIEW_PRIORITY Prioritize for ma

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks and leakage check

Some high-ranked pages may be weak recommendations because the baseline score uses only a small number of available signals. A page may have low impressions or search volume for a legitimate reason and may not actually need a content refresh.

The baseline does not use future performance windows, product flags, or the observed trend label to calculate the score. The score is therefore intended as a simple baseline for manual review rather than a final ML system.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-07 Section 4
# Inspect weak picks and perform a basic leakage check

print("WEAK PICK CHECK")
print("=" * 60)

# Pages with high baseline score but relatively low search volume
weak_picks = output[
    (output["baseline_score"] >= output["baseline_score"].quantile(0.90)) &
    (output["search_volume"] <= output["search_volume"].median())
]

print(
    "High-score pages with below-median search volume:",
    len(weak_picks)
)

print("\nExample weak picks:")
print(
    weak_picks.head(10).to_string(index=False)
)

# --------------------------------------------------
# Leakage check
# --------------------------------------------------

print("\nLEAKAGE CHECK")
print("=" * 60)

leakage_keywords = [
    "trend",
    "future",
    "product",
    "label",
    "target",
    "cheat"
]

possible_leakage = [
    col for col in df.columns
    if any(word in col.lower() for word in leakage_keywords)
]

print("Potentially sensitive columns found in dataframe:")
print(possible_leakage)

print("\nColumns actually used in baseline score:")
print([
    "search_volume",
    "impressions_90d"
])

print(
    "\nLeakage conclusion: "
    "The baseline score does not directly use trend_direction, "
    "future outcome labels, or product flags."
)

WEAK PICK CHECK
High-score pages with below-median search volume: 3396

Example weak picks:
 rank  baseline_score     reason_code  search_volume  impressions_90d
    1        0.932037 REVIEW_PRIORITY            0.0                1
    2        0.932037 REVIEW_PRIORITY            0.0                1
    3        0.932037 REVIEW_PRIORITY            0.0                1
    4        0.932037 REVIEW_PRIORITY            0.0                1
    5        0.932037 REVIEW_PRIORITY            0.0                1
    6        0.932037 REVIEW_PRIORITY            0.0                1
    7        0.932037 REVIEW_PRIORITY            0.0                1
    8        0.932037 REVIEW_PRIORITY            0.0                1
    9        0.932037 REVIEW_PRIORITY            0.0                1
   10        0.932037 REVIEW_PRIORITY            0.0                1

LEAKAGE CHECK
Potentially sensitive columns found in dataframe:
['trend_direction', 'trend_pct']

Columns actually used in baseline score

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.